# 03. Toy GPT: from Embedding to Next-Token Prediction

这个 notebook 不训练一个真正的 GPT，因为训练语言模型需要大量文本和算力。

我们的目标更小：用一个 toy vocabulary 和一个很小的 Transformer decoder，把 GPT 的核心计算链条跑通：

```text
tokens -> token embedding + position embedding
       -> Q, K, V
       -> causal self-attention
       -> Transformer block
       -> logits
       -> softmax
       -> next token
```

看完以后，应该能回答两个问题：

1. GPT 在一次 forward pass 里到底算了什么？
2. GPT 训练和生成时，输入输出是如何组织的？

## 1. 准备一个很小的词表

真实 GPT 的 vocabulary 通常是几万到十几万个 token。这里我们只用 8 个 token，方便看清楚张量形状。

一个 token 可以是一个词、一个词的一部分、一个标点，或者特殊符号。这里为了简单，直接把每个英文单词当作一个 token。

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(42)

DEVICE = "cpu"
D_MODEL = 16      # token vector dimension
BLOCK_SIZE = 8    # maximum sequence length in this toy model

vocab = ["<bos>", "cell", "gene", "rna", "protein", "data", "learns", "<eos>"]
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

VOCAB_SIZE = len(vocab)

prompt_tokens = ["<bos>", "cell", "gene", "rna"]
input_ids = torch.tensor([[stoi[token] for token in prompt_tokens]], dtype=torch.long, device=DEVICE)

print("vocab size:", VOCAB_SIZE)
print("input tokens:", prompt_tokens)
print("input_ids shape:", tuple(input_ids.shape))
print(input_ids)

## 2. Token Embedding 和 Position Embedding

GPT 不能直接处理字符串。第一步是把 token id 变成向量。

设 batch size 为 $B$，序列长度为 $T$，embedding 维度为 $D$。

- `input_ids.shape = (B, T)`
- `token_embedding(input_ids).shape = (B, T, D)`
- `position_embedding(positions).shape = (T, D)`

位置向量很重要，因为 self-attention 本身不天然知道 token 的顺序。

In [ ]:
token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
position_embedding = nn.Embedding(BLOCK_SIZE, D_MODEL)

B, T = input_ids.shape
positions = torch.arange(T, device=DEVICE)

token_vectors = token_embedding(input_ids)
position_vectors = position_embedding(positions)
x = token_vectors + position_vectors.unsqueeze(0)

print("token_vectors shape:", tuple(token_vectors.shape))
print("position_vectors shape:", tuple(position_vectors.shape))
print("x shape:", tuple(x.shape))

## 3. 从 $x$ 计算 Q, K, V

self-attention 会把每个位置的向量 $x_t$ 投影成三个向量：

- Query: $q_t$，表示当前位置想找什么信息。
- Key: $k_t$，表示当前位置能被别人如何匹配。
- Value: $v_t$，表示当前位置真正提供给别人汇总的信息。

在代码里，这三个投影通常就是三个线性层：

```python
q = W_q(x)
k = W_k(x)
v = W_v(x)
```

`AI 使用指引`：如果这里不清楚，可以问 AI：“用一个不超过 4 个 token 的例子解释 Q、K、V 分别是什么，为什么 attention score 要用 Q 和 K 的点积？” 理解到能说出 Q/K 决定注意力权重，V 是被加权汇总的信息即可。

In [ ]:
W_q = nn.Linear(D_MODEL, D_MODEL, bias=False)
W_k = nn.Linear(D_MODEL, D_MODEL, bias=False)
W_v = nn.Linear(D_MODEL, D_MODEL, bias=False)

q = W_q(x)
k = W_k(x)
v = W_v(x)

attention_scores = q @ k.transpose(-2, -1) / math.sqrt(D_MODEL)

print("q shape:", tuple(q.shape))
print("k shape:", tuple(k.shape))
print("v shape:", tuple(v.shape))
print("attention_scores shape:", tuple(attention_scores.shape))

## 4. Causal Mask: GPT 只能看左边

GPT 是自回归模型。预测第 $t$ 个位置的下一个 token 时，只允许使用当前位置以及它左边的信息，不能偷看右边未来的 token。

所以 attention score 要加上 causal mask：

- 允许看的位置：保留 score。
- 不允许看的位置：设成一个很大的负数，例如 `-inf`。

这样 softmax 之后，未来位置的注意力权重就会变成 0。

这里还有一个很重要的工程原因：causal mask 让训练可以并行。训练时我们已经知道整段文本，所以可以一次把所有位置的 token 都送进模型，同时计算每个位置的 next-token loss。mask 保证第 $t$ 个位置虽然和其他位置一起计算，但它不会看到右边的未来 token。

当然，也可以写一个循环：先算第 1 个位置，再算第 2 个位置，再算第 3 个位置。这样概念上更直接，但效率很低，GPU 很难被充分利用。Transformer 的做法是把整个序列组织成矩阵运算，用 causal mask 维持自回归约束。

In [ ]:
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=DEVICE), diagonal=1)
masked_scores = attention_scores.masked_fill(causal_mask, float("-inf"))
attention_weights = F.softmax(masked_scores, dim=-1)
context = attention_weights @ v

print("causal mask, True means forbidden:")
print(causal_mask.int())
print("attention_weights shape:", tuple(attention_weights.shape))
print("context shape:", tuple(context.shape))

for row_idx, token in enumerate(prompt_tokens):
    weights = attention_weights[0, row_idx].detach()
    readable = {prompt_tokens[col_idx]: round(float(weights[col_idx]), 3) for col_idx in range(T)}
    print(f"{token:>5} attends to:", readable)

## 5. 把 Attention 放进一个 Transformer Block

一个最小的 GPT block 通常包含：

1. masked self-attention
2. residual connection
3. layer normalization
4. feed-forward network, 这里用一个很小的 MLP

真实模型通常使用 multi-head attention 和更复杂的激活函数。这里先用单头 attention 和 ReLU，重点是看清楚结构。

这里的 residual connection 指的是把某一层的输入直接加到这一层的输出上，例如代码里的：

```python
x = x + self.attn(self.ln1(x))
x = x + self.ffn(self.ln2(x))
```

直观地说，模块只需要学习“在原来的表示上应该改多少”，而不是每一层都从零重写整个表示。残差连接最有代表性的来源是 ResNet 论文 *Deep Residual Learning for Image Recognition*，He et al., 2015。后来 Transformer 也大量使用这种结构，使很深的网络更容易训练。

LayerNorm 可以理解成在每个 token 的 hidden vector 内部做一次尺度整理。它不改变序列长度，也不负责“记住知识”，主要作用是让进入 attention 或 FFN 的数值范围更稳定。直觉上，如果每一层输出的数值大小都飘来飘去，后面的层就很难训练；LayerNorm 像是在每个小模块前把表示调整到更容易处理的范围。这里不需要记具体公式，先记住它是稳定深层 Transformer 训练的重要组件即可。

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        scores = q @ k.transpose(-2, -1) / math.sqrt(D)
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        scores = scores.masked_fill(causal_mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        context = weights @ v
        return self.out_proj(context)


class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


block = TinyTransformerBlock(D_MODEL)
block_output = block(x)

print("block input shape:", tuple(x.shape))
print("block output shape:", tuple(block_output.shape))

## 6. Tiny GPT: 输出每个位置的下一个 token 分数

GPT 的最后一步会把 hidden state 映射回 vocabulary 大小：

```text
hidden state shape: (B, T, D)
logits shape:       (B, T, V)
```

其中 $V$ 是 vocabulary size。`logits[b, t]` 是模型在第 `t` 个位置预测下一个 token 的原始分数。

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.block = TinyTransformerBlock(d_model)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}.")

        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions).unsqueeze(0)
        x = self.block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits


model = TinyGPT(VOCAB_SIZE, D_MODEL, BLOCK_SIZE).to(DEVICE)
logits = model(input_ids)

print("input_ids shape:", tuple(input_ids.shape))
print("logits shape:", tuple(logits.shape))

## 7. Softmax 得到下一个 token 的概率

我们通常只拿最后一个位置的 logits 来预测下一个 token。

注意：模型现在没有训练过，所以概率没有实际语言意义。这里的目的只是看清楚计算流程。

In [ ]:
next_logits = logits[:, -1, :]
next_probs = F.softmax(next_logits, dim=-1)

top_probs, top_ids = torch.topk(next_probs[0], k=5)

print("Prompt:", " ".join(prompt_tokens))
print("Top next-token probabilities from the untrained model:")
for prob, token_id in zip(top_probs, top_ids):
    print(f"{itos[int(token_id)]:>8}: {float(prob.detach()):.3f}")

predicted_id = int(torch.argmax(next_probs, dim=-1)[0])
print("argmax next token:", itos[predicted_id])

## 8. GPT 是怎么训练的：shifted next-token prediction

训练 GPT 时，一段 token 序列会被拆成：

```text
input:  <bos> cell gene rna
target: cell  gene rna  protein
```

也就是说，每个位置都预测它右边的下一个 token。

如果 `logits.shape = (B, T, V)`，`targets.shape = (B, T)`，PyTorch 的 cross entropy 需要把它们整理成：

```python
logits_for_loss.shape = (B * T, V)
targets_for_loss.shape = (B * T,)
```

`AI 使用指引`：如果 shifted target 不直观，可以问 AI：“给我一个长度为 5 的 token 序列，逐位置列出 GPT 训练时每个输入位置对应的 target 是什么。” 理解到 GPT 不是一次只预测最后一个 token，而是对每个位置都做 next-token prediction 即可。

In [ ]:
training_tokens = ["<bos>", "cell", "gene", "rna", "protein"]
training_ids = torch.tensor([[stoi[token] for token in training_tokens]], dtype=torch.long, device=DEVICE)

train_inputs = training_ids[:, :-1]
train_targets = training_ids[:, 1:]

train_logits = model(train_inputs)
loss = F.cross_entropy(
    train_logits.reshape(-1, VOCAB_SIZE),
    train_targets.reshape(-1),
)

print("training tokens:", training_tokens)
print("train_inputs:", [[itos[int(i)] for i in row] for row in train_inputs])
print("train_targets:", [[itos[int(i)] for i in row] for row in train_targets])
print("train_logits shape:", tuple(train_logits.shape))
print("loss from the untrained model:", round(float(loss.detach()), 4))

真正训练时，会在大量文本上重复下面的过程：

```python
logits = model(input_ids)
loss = cross_entropy(logits, targets)
loss.backward()
optimizer.step()
optimizer.zero_grad()
```

训练完成后，模型参数会让正确 next token 的概率更高。

注意训练时可以并行：同一段序列中的所有位置可以一起计算 logits 和 loss。原因是训练数据里 target 已经存在，模型不需要先生成前一个 token，才能知道下一个训练目标。causal mask 负责防止信息泄漏，所以并行计算仍然符合“只能看左边”的规则。

这个 notebook 不运行训练循环，因为即使是很小的语言模型，真正训练也需要比这里多得多的数据和时间。

## 9. GPT 是怎么生成的：一次生成一个 token

生成时没有 target。模型从 prompt 开始，重复：

1. 用当前所有 token 做 forward pass。
2. 取最后一个位置的 logits。
3. softmax 得到下一个 token 的概率。
4. 选出或采样一个 token。
5. 把新 token 接到序列末尾。

推理生成时通常不能像训练那样在时间维度上完全并行。原因是第 $t+1$ 个 token 的输入依赖第 $t$ 步刚生成出来的 token；在生成出来之前，后面的 token 还不存在。因此生成必须一步一步往右扩展。真实系统会用 KV cache 等方法减少重复计算，但自回归生成的顺序依赖仍然存在。

`AI 使用指引`：如果 generation loop 不清楚，可以问 AI：“为什么 GPT 生成文本必须把刚生成的 token 再喂回模型？用三步以内的例子说明。” 理解到生成是 autoregressive，一次只扩展一个 token 即可。

In [ ]:
@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens, temperature=1.0):
    model.eval()
    generated = prompt_ids.clone()

    for _ in range(max_new_tokens):
        # Keep only the most recent block_size tokens if the sequence gets too long.
        model_input = generated[:, -model.block_size :]
        logits = model(model_input)
        next_logits = logits[:, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_id], dim=1)

        if int(next_id[0, 0]) == stoi["<eos>"]:
            break

    return generated


prompt = ["<bos>", "cell"]
prompt_ids = torch.tensor([[stoi[token] for token in prompt]], dtype=torch.long, device=DEVICE)
generated_ids = generate(model, prompt_ids, max_new_tokens=6, temperature=1.0)
generated_tokens = [itos[int(token_id)] for token_id in generated_ids[0]]

print("generated token ids:", generated_ids.tolist())
print("generated tokens:", generated_tokens)

## 10. 小结

这个 toy GPT 做了三件事：

1. 把 token id 变成 token embedding，并加上 position embedding。
2. 用 causal self-attention 和 MLP 组成 Transformer block。
3. 把 hidden state 映射成 vocabulary logits，再用 softmax 得到 next-token 概率。

需要记住的区别：

- 训练时：已知完整文本，把序列右移一位作为 target，用 causal mask 避免偷看未来，因此可以并行计算所有位置的 loss。
- 生成时：没有 target，从 prompt 开始，一次预测并接上一个新 token；后面的输入依赖前一步生成结果，所以时间维度上不能完全并行。

这个模型没有训练，所以生成内容没有语义。真正 GPT 的关键区别主要在于：更大的 vocabulary、更多数据、更大的 embedding 维度、更多 Transformer blocks、multi-head attention，以及长时间训练。